# 第4回：データ探偵—分布・欠損・外れ値

**今日の問い：モデルを作る前に、データの怪しいところをどう見つけるか。**

上から順に実行してください。`TRY`は全員、`CHANGE`は値を1つ変える練習、
`CHALLENGE`は余裕がある人向けです。`DEEP DIVE`は経験者や自習向けの発展です。
分からないコードは、セル全体ではなく気になる数行をM365 Copilotへ貼って相談します。


In [ ]:
from pathlib import Path

def find_repo_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("pyproject.tomlがある勉強会フォルダ内で実行してください")

ROOT = find_repo_root()
DATA = ROOT / "data"
print("教材フォルダ:", ROOT)


## この回でできるようになること

- 単変量・二変量・群別の順でデータを見る
- 欠損の発生機構と外れ値を、検定や多変量手法で客観的に調べる
- 図と統計量から、断定ではなく検証可能な仮説を作る

### 進み方

`CORE`は同期90分で扱う本線、`DEEP DIVE`は時間があれば扱う深掘り、
`SELF-STUDY`は任意自習です。すべて終わらなくても次回へ進めます。
経験者は`CORE`を早めに終え、`DEEP DIVE`を5人で分担して読むと深まります。

### 先に押さえる言葉

- 分布：値がどこにどれだけ存在するか
- 外れ値：他と大きく異なる観測値
- 相互情報量：非線形も捉える関連の強さ
- 欠損機構：MCAR/MAR/MNARという欠損の起こり方
- 多変量外れ値：単変量では見えない組み合わせの異常

> **実行前の30秒予想**：今日の問いに、今の言葉で仮の答えを書いてから始めます。


In [ ]:
import pandas as pd

df = pd.read_csv(DATA / "compound_experiments.csv")
print(f"{len(df)}行 × {len(df.columns)}列")
df.head()


In [ ]:
import matplotlib.pyplot as plt
from matplotlib import font_manager
for _name in ["Yu Gothic", "Meiryo", "Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic"]:
    if _name in {f.name for f in font_manager.fontManager.ttflist}:
        plt.rcParams["font.family"] = _name
        break
import seaborn as sns
sns.set_theme(style="whitegrid")


## TRY：1変数の分布を見る


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(data=df, x="yield_pct", bins=20, ax=axes[0])
axes[0].set_title("収率の分布")
sns.boxplot(data=df, x="reaction_time_h", ax=axes[1])
axes[1].set_title("反応時間：外れ値候補を探す")
plt.tight_layout()


## 2変数の関係とカテゴリ比較


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.scatterplot(data=df, x="temperature_c", y="yield_pct", hue="catalyst", alpha=0.65, ax=axes[0])
axes[0].set_title("温度と収率")
sns.boxplot(data=df, x="catalyst", y="yield_pct", ax=axes[1])
axes[1].set_title("触媒別の収率")
plt.tight_layout()


## TRY：欠損と怪しい値を表で確認


In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
display(missing[missing > 0].to_frame("欠損数"))
display(df.nlargest(5, "temperature_c")[["sample_id", "temperature_c", "reaction_time_h", "yield_pct"]])


## CHANGE

色分けを`catalyst`から`solvent`へ変えます。

## 注意

外れ値は入力ミスとは限りません。「誰に確認するか」「残す場合に何が起きるか」まで考えます。


## DEEP DIVE：相関・相互情報量・欠損機構・多変量外れ値

図の印象を、統計量と手法で裏づけます。


In [ ]:
numeric_cols = ["temperature_c", "reaction_time_h", "concentration_m", "molecular_weight", "logp", "tpsa", "yield_pct"]
correlation = df[numeric_cols].corr()
plt.figure(figsize=(8, 5))
sns.heatmap(correlation, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("数値列の相関（因果ではない）")
plt.tight_layout()


### 相関では見えない関係を相互情報量で拾う

温度は最適点で収率が最大になる山型のため、直線的な相関は弱くても相互情報量は大きく出ます。


In [ ]:
from sklearn.feature_selection import mutual_info_regression

mi_source = ["temperature_c", "reaction_time_h", "concentration_m", "molecular_weight", "logp", "tpsa"]
mi_frame = df[mi_source + ["yield_pct"]].dropna()
mi = mutual_info_regression(mi_frame[mi_source], mi_frame["yield_pct"], random_state=42)
pearson = mi_frame[mi_source].corrwith(mi_frame["yield_pct"]).abs()
compare = pd.DataFrame({"相互情報量": mi, "|相関|": pearson.to_numpy()}, index=mi_source)
compare.sort_values("相互情報量", ascending=False).round(3)


### 欠損の起こり方を疑う

欠損率が他の列で偏るなら、ランダムでない欠損（MAR）を疑います。


In [ ]:
miss = df.assign(temp_missing=df["temperature_c"].isna())
by_solvent = miss.groupby("solvent", dropna=False)["temp_missing"].mean().round(3)
print("溶媒別の温度欠損率:")
print(by_solvent)
print("溶媒でほぼ一定ならMCARに近い。偏るならMARを疑う。")


### 多変量外れ値

単変量では正常でも、組み合わせが異常な試料をIsolationForestで探します。


In [ ]:
from sklearn.ensemble import IsolationForest

iso_cols = ["temperature_c", "reaction_time_h", "concentration_m", "yield_pct"]
iso_data = df[iso_cols].fillna(df[iso_cols].median())
flags = IsolationForest(contamination=0.03, random_state=42).fit_predict(iso_data)
outliers = df.loc[flags == -1, ["sample_id", *iso_cols]]
print("多変量外れ値候補:", len(outliers), "件")
outliers.round(2)


## よくある誤り

- 外れ値を自動削除する
- 相関を因果と読む
- 見栄えの良い図だけを選ぶ

## SELF-STUDY（任意・30〜60分）

- 相互情報量の上位3列について、散布図で関係の形を確認する
- IsolationForestの外れ値候補2件を、残す場合と除く場合で整理する

成果は完成したコードでなくても、予想・変更点・出力・解釈を4行で残せば十分です。

## 振り返りチェック

1. 相関係数と相互情報量はどう違うか
2. MCARとMARの違いは何か
3. 多変量外れ値が単変量で見つからない理由は何か

答えに詰まった項目が、次に見返す場所です。暗記ではなくNotebookの該当セルを指せればOKです。
